# Face Shape CNN — Transfer Learning Notebook

The earlier approach measured three hand-picked ratios (`hr`, `fr`, `jr`) from face landmarks and fed those three numbers to a Random Forest — that topped out around 45% accuracy. Three numbers just don't carry enough information to tell five face shapes apart reliably.

This notebook trains a **Convolutional Neural Network (CNN)** directly on the face photos instead. A CNN looks at the raw pixels and learns its own features — edges, curves, proportions — rather than us guessing which three ratios matter.

**Transfer learning**, in plain terms: instead of teaching a network to see from scratch (which needs millions of images), we start from **MobileNetV2**, a network Google already trained on 1.4 million general photos (ImageNet). It already knows how to recognize edges, textures, and shapes. We freeze that knowledge and just teach a new small "head" on top to map those learned features to our 5 face shapes: Heart, Oblong, Oval, Round, Square. This works well even with only ~5,000 training photos.

> **ARCHIVED** — superseded by `train_face_shape_cnn_v2.ipynb` (fixed framing/preprocessing bugs found by testing this version's output). The live app's `face_shape_cnn.keras`/`.tflite` were trained with v2, not this notebook. Kept for reference only.

## Step 1: Confirm the GPU is on

CNN training is much faster on a GPU than a CPU. Before doing anything else, check that Colab actually gave us one. If the cell below prints "No GPU found", go to **Runtime → Change runtime type → Hardware accelerator → GPU (T4)**, then re-run this cell.

In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"GPU found: {gpus[0]}")
else:
    print("No GPU found! Go to Runtime > Change runtime type > "
          "select GPU, then re-run this cell.")

print("TensorFlow version:", tf.__version__)

## Step 2: Get the dataset into Colab

The dataset has ~5,000 photos, which is too much to reliably upload through the browser's file picker every session (it can time out or fail partway). The more reliable route is:

1. On your computer, zip the `face_shape_dataset` folder (so the zip contains `face_shape_dataset/training_set/...` and `face_shape_dataset/testing_set/...`).
2. Upload that one zip file to your Google Drive (e.g. into "My Drive"), just once.
3. Run the cell below — it mounts your Drive (Colab will ask you to log in and grant access) and unzips the dataset onto Colab's local disk. Training reads much faster from local disk than directly off Drive.

**Edit `ZIP_PATH` below** to match wherever you placed the zip in your Drive.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

# Change this to wherever you uploaded the zip in your Google Drive
ZIP_PATH = '/content/drive/MyDrive/face_shape_dataset.zip'

In [ ]:
import os

EXTRACT_DIR = '/content/dataset'

if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(
        f"Couldn't find {ZIP_PATH} — check the path matches where you "
        "uploaded the zip in Google Drive."
    )

os.makedirs(EXTRACT_DIR, exist_ok=True)
!unzip -q "{ZIP_PATH}" -d {EXTRACT_DIR}


def find_dataset_root(root):
    # Zips vary in whether they include the top folder name or not,
    # so search for the first place that has both split folders,
    # instead of assuming one fixed path.
    for dirpath, dirnames, _ in os.walk(root):
        if 'training_set' in dirnames and 'testing_set' in dirnames:
            return dirpath
    raise FileNotFoundError(
        "Couldn't find training_set/ and testing_set/ folders anywhere "
        "inside the extracted zip. Check the zip's folder structure."
    )


DATASET_ROOT = find_dataset_root(EXTRACT_DIR)
TRAIN_DIR    = os.path.join(DATASET_ROOT, 'training_set')
TEST_DIR     = os.path.join(DATASET_ROOT, 'testing_set')

print("Dataset root:", DATASET_ROOT)
print("Training folder:", TRAIN_DIR)
print("Testing folder:", TEST_DIR)

## Step 3: Sanity-check the folder counts

Before training on anything, print how many photos landed in each class folder, for both splits. This is just a visual check that the unzip worked and nothing is empty or misnamed.

In [ ]:
SHAPES = ['Heart', 'Oblong', 'Oval', 'Round', 'Square']

for split_name, split_dir in [('training_set', TRAIN_DIR), ('testing_set', TEST_DIR)]:
    print(f"{split_name}:")
    for shape in SHAPES:
        folder = os.path.join(split_dir, shape)
        count = len(os.listdir(folder)) if os.path.isdir(folder) else 0
        print(f"  {shape}: {count}")

## Step 4: Remove corrupted images

The crash (`jpeg::Uncompress failed`) happens because a handful of the ~5,000 photos aren't valid, fully-readable JPEGs — TensorFlow's decoder gives up partway through training instead of skipping them. Checking file extensions isn't enough, since a file can be named `.jpg` and still be truncated or corrupted inside.

This cell opens every single image with PIL and actually tries to decode it (not just peek at the header). Anything that fails gets moved — not deleted, so nothing is lost by mistake — into a `corrupted_removed/` folder that mirrors the original split/shape structure. Only after this cleanup do we move on to loading images for training.

In [ ]:
import shutil
from collections import Counter
from PIL import Image

CORRUPTED_DIR = '/content/corrupted_removed'


def is_valid_image(path):
    try:
        with Image.open(path) as img:
            img.verify()  # catches most corrupt/truncated files
        with Image.open(path) as img:
            img.load()  # fully decode pixels; verify() alone misses some truncated JPEGs
        return True
    except Exception:
        return False


checked = 0
removed = Counter()

for split_dir in [TRAIN_DIR, TEST_DIR]:
    split_name = os.path.basename(split_dir)
    for shape in SHAPES:
        folder = os.path.join(split_dir, shape)
        if not os.path.isdir(folder):
            continue
        for filename in os.listdir(folder):
            path = os.path.join(folder, filename)
            if not os.path.isfile(path):
                continue
            checked += 1
            if not is_valid_image(path):
                dest_dir = os.path.join(CORRUPTED_DIR, split_name, shape)
                os.makedirs(dest_dir, exist_ok=True)
                shutil.move(path, os.path.join(dest_dir, filename))
                removed[shape] += 1

print(f"Checked {checked} images.")
print(f"Removed {sum(removed.values())} corrupted/unreadable images.\n")

print("Removed per face shape:")
for shape in SHAPES:
    print(f"  {shape}: {removed[shape]}")

print("\nFinal clean counts per face shape:")
for split_name, split_dir in [('training_set', TRAIN_DIR), ('testing_set', TEST_DIR)]:
    print(f"{split_name}:")
    for shape in SHAPES:
        folder = os.path.join(split_dir, shape)
        count = len(os.listdir(folder)) if os.path.isdir(folder) else 0
        print(f"  {shape}: {count}")

## Step 5: Load images and set up augmentation

`image_dataset_from_directory` reads images straight from the folder structure and uses each subfolder name (Heart, Oblong, ...) as the label automatically — no manual CSV needed for this approach.

- **Resizing to 224×224**: this is the input size MobileNetV2 was trained on, so every photo gets resized to match.
- **`training_set` gets split again** into a training portion (85%) and a validation portion (15%). Validation photos are held out during training so we can check progress on unseen photos each epoch. `testing_set` stays completely separate and untouched until the very end, for the final honest accuracy number.
- **Data augmentation** (horizontal flip + a small random rotation) is applied only while training. It randomly tweaks each training photo slightly on the fly, so the model sees a bit more variety and doesn't just memorize the exact 5,000 images — this reduces overfitting.

This cell also prints two sanity checks:
1. **Class balance** for the train and validation splits specifically (not just the raw folders) — confirms the split didn't accidentally skew toward one face shape.
2. **Preprocessing proof** — it prints the actual pixel range before and after `mobilenet_v2.preprocess_input`, so you can see with real numbers that it's not a generic `1/255` rescale (that would show `[0.0, 1.0]`); MobileNetV2's own preprocessing maps pixels to roughly `[-1, 1]`.

In [ ]:
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
SEED       = 42  # fixed seed so the train/validation split is reproducible

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=0.15,
    subset='training',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=0.15,
    subset='validation',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    shuffle=False,  # keep order so predictions line up with true labels later
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)

class_names = train_ds.class_names
print("Classes:", class_names)

# Class balance check: confirms the train/validation split (and the
# corrupted-image cleanup before it) didn't leave any shape underrepresented.
def count_labels(dataset):
    counts = Counter()
    for _, labels in dataset.unbatch():
        counts[class_names[labels.numpy()]] += 1
    return counts

print("\nTrain split counts:     ", dict(count_labels(train_ds)))
print("Validation split counts:", dict(count_labels(val_ds)))

# Preprocessing proof: shows the actual pixel range before/after
# mobilenet_v2.preprocess_input, so this isn't just claimed but measured.
# A generic 1/255 rescale would show [0.0, 1.0] here instead.
sample_images, _ = next(iter(train_ds))
raw_min, raw_max = sample_images.numpy().min(), sample_images.numpy().max()
preprocessed = tf.keras.applications.mobilenet_v2.preprocess_input(sample_images)
pre_min, pre_max = preprocessed.numpy().min(), preprocessed.numpy().max()
print(f"\nRaw pixel range: [{raw_min:.1f}, {raw_max:.1f}]")
print(f"After mobilenet_v2.preprocess_input: [{pre_min:.2f}, {pre_max:.2f}] "
      f"(MobileNetV2 expects roughly [-1, 1] here)")

# Data augmentation: only ever applied to training images, and only
# during training (Keras automatically turns it off at prediction time).
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.05),  # up to ~18 degrees either way
])

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds   = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds  = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

## Step 6: Build the model — MobileNetV2 base + a new head

The model has two parts:

1. **Base: MobileNetV2**, loaded with its ImageNet-trained weights, and with `include_top=False` (we drop its original 1000-class ImageNet output layer, since we don't want those classes). We **freeze** it (`trainable = False`) so its existing knowledge isn't destroyed by large early gradient updates from our small, randomly-initialized head.
2. **Head: our new layers**, added on top — a pooling layer to condense MobileNetV2's output into one feature vector per image, a dropout layer (randomly turns off some neurons during training to reduce overfitting), and a final Dense layer with 5 outputs (one probability per face shape).

`preprocess_input` rescales pixel values into the range MobileNetV2 expects (roughly -1 to 1) — this must match how it was trained originally.

In [ ]:
NUM_CLASSES = len(class_names)
INITIAL_LR  = 1e-3  # standard Adam default; head starts from random weights so it can learn fast

base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights='imagenet',
)
base_model.trainable = False
print(f"MobileNetV2 base: {len(base_model.layers)} layers, all frozen for initial training.")

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=INITIAL_LR),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

model.summary()

## Step 7: Initial training (frozen base)

With the MobileNetV2 base frozen, we only train the small new head — this is fast and low-risk since there are far fewer trainable parameters.

**Early stopping**: instead of blindly running all 20 epochs, we watch validation accuracy. If it stops improving for 5 epochs in a row (`patience=5`), training stops early and the best-performing weights are restored. This avoids wasting time and avoids overfitting from training too long.

The cell prints exactly how many epochs actually ran and whether early stopping cut it short — if this number looks suspiciously small (e.g. 3-4 epochs), that's a sign `patience` may need to be increased.

In [ ]:
INITIAL_EPOCHS = 20

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    patience=5,
    restore_best_weights=True,
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=INITIAL_EPOCHS,
    callbacks=[early_stop],
)

ran_epochs = len(history.epoch)
print(f"\nInitial training learning rate: {INITIAL_LR:.1e}")
print(f"Initial training ran for {ran_epochs} of {INITIAL_EPOCHS} planned epochs "
      f"({'early stopping triggered' if ran_epochs < INITIAL_EPOCHS else 'ran full length, early stopping never triggered'}).")
print(f"Best validation accuracy (frozen base): {max(history.history['val_accuracy']):.2%}")

## Step 8: Fine-tuning — unfreeze the top layers

Now that the new head has learned something reasonable, we unfreeze the **last part** of MobileNetV2 (its deeper layers, which specialize in more complex/task-specific patterns) and let those retrain slightly, together with the head. We deliberately keep the earlier layers frozen — those already learned generic things like edges and textures that transfer fine as-is.

We also drop the learning rate a lot (from 0.001 to 0.00001). Fine-tuning pretrained weights needs small, careful updates — a large learning rate here would wreck the pretrained knowledge instead of gently adjusting it. This step is usually what pushes accuracy meaningfully higher than the frozen-base result alone.

The cell prints exactly how many MobileNetV2 layers got unfrozen, the learning rate used for this phase versus the initial phase, and how many epochs actually ran — so you can directly confirm this step is set up the way it's described here, not just take it on faith.

In [ ]:
base_model.trainable = True

# MobileNetV2 has ~154 layers; freeze the first 100 and fine-tune only
# the last ~54, which hold the more task-specific features.
FINE_TUNE_AT = 100
for layer in base_model.layers[:FINE_TUNE_AT]:
    layer.trainable = False

unfrozen = sum(1 for layer in base_model.layers if layer.trainable)
print(f"Fine-tuning: unfroze the last {unfrozen} of {len(base_model.layers)} MobileNetV2 layers.")

FINE_TUNE_LR = 1e-5  # much smaller than INITIAL_LR so we nudge pretrained weights, not overwrite them
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=FINE_TUNE_LR),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
print(f"Fine-tuning learning rate: {FINE_TUNE_LR:.1e} (vs {INITIAL_LR:.1e} used for initial training)")

FINE_TUNE_EPOCHS = 10
TOTAL_EPOCHS = history.epoch[-1] + 1 + FINE_TUNE_EPOCHS

fine_tune_early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    patience=5,
    restore_best_weights=True,
)

history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=TOTAL_EPOCHS,
    initial_epoch=history.epoch[-1] + 1,
    callbacks=[fine_tune_early_stop],
)

ran_epochs_fine = len(history_fine.epoch)
print(f"\nFine-tuning ran for {ran_epochs_fine} of {FINE_TUNE_EPOCHS} planned epochs "
      f"({'early stopping triggered' if ran_epochs_fine < FINE_TUNE_EPOCHS else 'ran full length, early stopping never triggered'}).")
print(f"Best validation accuracy (after fine-tuning): {max(history_fine.history['val_accuracy']):.2%}")

## Step 9: Plot accuracy and loss curves

These plots join both training phases (frozen-base, then fine-tuning) into one continuous timeline.

**How to read them for overfitting**: if training accuracy keeps climbing while validation accuracy flattens or drops (equivalently, training loss keeps falling while validation loss rises), the model is starting to memorize the training photos rather than learning general patterns — that's overfitting.

In [ ]:
import matplotlib.pyplot as plt

acc      = history.history['accuracy']      + history_fine.history['accuracy']
val_acc  = history.history['val_accuracy']  + history_fine.history['val_accuracy']
loss     = history.history['loss']          + history_fine.history['loss']
val_loss = history.history['val_loss']      + history_fine.history['val_loss']
fine_tune_start = history.epoch[-1] + 1

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(acc, label='Training accuracy')
ax1.plot(val_acc, label='Validation accuracy')
ax1.axvline(fine_tune_start, linestyle='--', color='gray', label='Fine-tuning starts')
ax1.set_title('Accuracy over epochs')
ax1.set_xlabel('Epoch')
ax1.legend()

ax2.plot(loss, label='Training loss')
ax2.plot(val_loss, label='Validation loss')
ax2.axvline(fine_tune_start, linestyle='--', color='gray', label='Fine-tuning starts')
ax2.set_title('Loss over epochs')
ax2.set_xlabel('Epoch')
ax2.legend()

plt.show()

## Step 10: Final evaluation on the held-out test set

This is the moment of truth: `testing_set/` was never used for training or validation, so accuracy here is the most honest estimate of how the model would perform on new photos.

Same three metrics as before:
- **Accuracy** — percentage correct overall.
- **Classification report** — precision/recall broken down per face shape.
- **Confusion matrix** — rows are the true shape, columns are the predicted shape, so you can see exactly which shapes get confused with each other.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

test_loss, test_accuracy = model.evaluate(test_ds)
print(f"\nFinal test accuracy: {test_accuracy:.2%}")

y_true = np.concatenate([labels.numpy() for _, labels in test_ds])
y_pred_probs = model.predict(test_ds)
y_pred = np.argmax(y_pred_probs, axis=1)

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
print("Confusion Matrix (rows = actual shape, columns = predicted shape):")
print(cm_df)

## Step 11: Save and download the model

We save as a single `.h5` file rather than TensorFlow's SavedModel format, which spreads files across a whole folder — a single file is much simpler to download reliably from Colab in one go.

In [ ]:
MODEL_FILENAME = 'face_shape_cnn_model.h5'

model.save(MODEL_FILENAME)
print(f"Model saved as {MODEL_FILENAME}")

In [ ]:
from google.colab import files

files.download(MODEL_FILENAME)